# Deep Learning: Churn Model Comparison

This notebook explores the add-on results generated by `scripts/train_deep_learning_model.py`. It keeps the existing research metrics separate from the newly trained hold-out experiment.

## Experimental protocol

The script uses one stratified customer-level train/validation/test split for all compared models, fits preprocessing on training rows only, and uses validation loss for MLP early stopping. The test set is reserved for final reporting. The default decision threshold is 0.50 and should be tuned on validation data if the business costs justify it.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "models").exists() and (ROOT.parent / "models").exists():
    ROOT = ROOT.parent
METRICS_PATH = ROOT / "models" / "deep_learning" / "comparison_metrics.json"
if not METRICS_PATH.exists():
    print("No results yet. Try: python -m scripts.train_deep_learning_model --data data/demo/telco_customer_churn_demo.csv --output-dir models/deep_learning")
else:
    experiment = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
    metrics = pd.DataFrame(experiment["metrics"]).T
    display(metrics.round(4))

## Compare ranking and classification metrics

ROC-AUC measures ranking across thresholds; PR-AUC is particularly useful with class imbalance. Precision and recall depend on the chosen threshold. Brier score and log loss assess probability quality. A model should not be declared better from one metric alone.

In [ ]:
if METRICS_PATH.exists():
    columns = [c for c in ["roc_auc", "pr_auc", "precision", "recall", "f1"] if c in metrics.columns]
    display(metrics[columns].plot(kind="bar", figsize=(10, 5), ylim=(0, 1), title="Classification metrics on the add-on hold-out set"))
    plt.tight_layout()
    plt.show()

## Inspect ROC and precision-recall curves

In [ ]:
if METRICS_PATH.exists():
    fig, ax = plt.subplots(figsize=(7, 5))
    for model_name, curve in experiment.get("roc_curves", {}).items():
        ax.plot(curve["fpr"], curve["tpr"], label=model_name)
    ax.plot([0, 1], [0, 1], linestyle="--", label="Chance baseline")
    ax.set(xlabel="False positive rate", ylabel="True positive rate", title="ROC curves")
    ax.legend()
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 5))
    for model_name, curve in experiment.get("precision_recall_curves", {}).items():
        ax.plot(curve["recall"], curve["precision"], label=model_name)
    ax.set(xlabel="Recall", ylabel="Precision", title="Precision-recall curves")
    ax.legend()
    plt.tight_layout()
    plt.show()

## Research notes to complete after the real experiment

- Record the exact source dataset, row count, class balance, train/validation/test sizes, and random seed.
- Compare model errors and probability calibration, not only aggregate metrics.
- If using the hybrid MLP, document how support tickets were linked to customers and how each text feature was cut off before the churn prediction date.
- Do not paste generated-fixture test results into the paper. Report only results from the actual dataset and held-out experiment.